In [29]:
!pip install librosa numpy pandas matplotlib scikit-learn tensorflow soundfile --progress-bar off


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import os
import librosa
import numpy as np

dataset_path = "dataset"

files = []

for actor in os.listdir(dataset_path):
    actor_path = os.path.join(dataset_path, actor)
    
    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                files.append(os.path.join(actor_path, file))

print("Total audio files:", len(files))
print("Example file:", files[0])

Total audio files: 1440
Example file: dataset\Actor_01\03-01-01-01-01-01-01.wav


In [31]:
def extract_features(file_path):
    
    audio, sample_rate = librosa.load(file_path, duration=3, offset=0.5)
    
    mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    
    mfcc_scaled = np.mean(mfcc.T, axis=0)
    
    return mfcc_scaled

In [32]:
test_file = files[0]

features = extract_features(test_file)

print("Feature vector length:", len(features))
print(features[:10])

Feature vector length: 40
[-6.7019543e+02  6.5063850e+01  8.8895434e-01  1.4715979e+01
  9.1821651e+00  6.6057485e-01 -3.8468359e+00 -3.5839462e+00
 -1.2959006e+01 -3.3001330e+00]


In [33]:
X = []
y = []

for file in files:
    
    # extract features
    features = extract_features(file)
    X.append(features)
    
    # extract emotion label from filename
    file_name = os.path.basename(file)
    
    emotion_code = file_name.split("-")[2]
    
    emotion_dict = {
        "01": "neutral",
        "02": "calm",
        "03": "happy",
        "04": "sad",
        "05": "angry",
        "06": "fearful",
        "07": "disgust",
        "08": "surprised"
    }
    
    emotion = emotion_dict[emotion_code]
    y.append(emotion)

print("Total samples:", len(X))
print("Total labels:", len(y))
print("Example label:", y[0])

Total samples: 1440
Total labels: 1440
Example label: neutral


In [34]:
X = np.array(X)
y = np.array(y)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (1440, 40)
Shape of y: (1440,)


In [35]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print("Encoded labels example:", y_encoded[:10])

import joblib
joblib.dump(encoder, "models/label_encoder.pkl")
print("Label encoder saved successfully!")

Encoded labels example: [5 5 5 5 1 1 1 1 1 1]
Label encoder saved successfully!


In [36]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = scaler.fit_transform(X)

import joblib
joblib.dump(scaler, "models/scaler.pkl")

['models/scaler.pkl']

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

Training samples: (1152, 40)
Testing samples: (288, 40)


In [38]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential()

model.add(Dense(256, activation='relu', input_shape=(40,)))
model.add(Dropout(0.3))

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))

model.add(Dense(len(set(y_encoded)), activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\Shaziya\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 256)            │        10,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,168 (203.78 KB)

 Trainable params: 52,168 (203.78 KB)

 Non-trainable params: 0 (0.00 B)

In [39]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.2092 - loss: 1.9860 - val_accuracy: 0.2951 - val_loss: 1.8428
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3125 - loss: 1.8040 - val_accuracy: 0.3854 - val_loss: 1.6818
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3793 - loss: 1.6723 - val_accuracy: 0.4340 - val_loss: 1.5537
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4271 - loss: 1.5402 - val_accuracy: 0.4688 - val_loss: 1.4575
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4722 - loss: 1.4223 - val_accuracy: 0.4931 - val_loss: 1.3863
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4870 - loss: 1.3409 - val_accuracy: 0.4792 - val_loss: 1.3486
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5278 - loss: 1.2730 - val_accuracy: 0.5139 - val_loss: 1.3091
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5703 - loss: 1.1821 - val_accuracy: 0.5035 - val_loss

In [40]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6701 - loss: 1.1824 
Test Accuracy: 0.6701388955116272


In [41]:
model.save("models/emotion_model.h5")

print("Model saved successfully!")

Model saved successfully!


In [42]:
print("Classes:", encoder.classes_)

Classes: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']
